# Samind — transformer training, end to end

One session: data checks → train candidate models → evaluate with slices → convert to TFLite (int8) → parity gate → artifacts back to Drive.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 is enough).
2. Dataset CSV in Drive at `MyDrive/samind/corpus.csv` — columns `text,label` (+ optional `source,comment,lang,category`).
3. Everything is parameterized in the CONFIG cell.

Budget: ~15–25 min per candidate on a T4 at 3–5k examples. Runs top to bottom unattended.

## 1. Setup

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
print(gpu or 'NO GPU — switch runtime type before continuing')
assert gpu, 'GPU runtime required'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone --depth 1 https://github.com/JadeRemi/samind-disorder.git /content/samind-disorder
%cd /content/samind-disorder/ml
# pinned: Colab's preinstalled transformers moves fast and breaks tokenizer loading
!pip -q install "transformers==4.46.3" "datasets==3.1.0" accelerate sentencepiece

In [ ]:
# ---------------- CONFIG ----------------
DATA = '/content/drive/MyDrive/samind/corpus.csv'   # labeled dataset
OUT = '/content/drive/MyDrive/samind/artifacts'      # results land here

# WordPiece-based (BERT family) only — that's what the Android tokenizer speaks
CANDIDATES = [
    'distilbert-base-multilingual-cased',
]

SMOKE_TEST = True   # True = quick pipeline check on any data size (throwaway model!)
                    # False = real training run with the task-card gates

EPOCHS = 1 if SMOKE_TEST else 4
MIN_EXAMPLES = 50 if SMOKE_TEST else 1000
LEARNING_RATE = 2e-5
SEQ_LEN = 128
SEED = 13

import os
os.makedirs(OUT, exist_ok=True)
print('SMOKE TEST — model is throwaway' if SMOKE_TEST else 'REAL RUN')

## 2. Data checks — fail fast before burning GPU time

In [ ]:
from samind_ml.dataset import load

df = load(DATA)
risky, safe = int(df.label.sum()), int((df.label == 0).sum())
print(f'{len(df)} unique examples: {risky} risky / {safe} safe')
assert len(df) >= MIN_EXAMPLES, f'need {MIN_EXAMPLES}+ examples, have {len(df)} — keep collecting'
gap = abs(risky - safe) / len(df)
if gap > 0.10:
    print(f'WARNING: class gap {gap:.0%} (target <=10%) — expect biased errors')

In [ ]:
!python -m samind_ml.report --data "$DATA"
!python -m samind_ml.split --data "$DATA" --out data/splits/ --seed 13

## 3. Baseline reference — the number to beat

In [ ]:
!python -m samind_ml.baseline --data "$DATA" --out artifacts/baseline/ --model logreg
!python -m samind_ml.baseline --data "$DATA" --out artifacts/baseline/

## 4. Train candidates

In [ ]:
import subprocess, sys

for name in CANDIDATES:
    slug = name.split('/')[-1]
    print(f'===== training {name}')
    r = subprocess.run([
        sys.executable, '-m', 'samind_ml.train',
        '--data', DATA,
        '--out', f'artifacts/{slug}',
        '--epochs', str(EPOCHS),
        '--lr', str(LEARNING_RATE),
        '--model-name', name,
    ], capture_output=True, text=True)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print('----- ERROR LOG -----')
        print(r.stderr[-4000:])
        raise SystemExit(f'training failed for {name}')

## 5. Evaluate — sliced metrics on the held-out test split

In [ ]:
import numpy as np, pandas as pd, torch
from pathlib import Path
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from samind_ml import evaluate
from samind_ml.normalize import normalize

test = pd.read_csv('data/splits/test.csv')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def score_checkpoint(ckpt, base_name, texts, batch=32):
    # tokenizer from the base model: training never changes it, and base repos
    # always load cleanly regardless of what the checkpoint dir contains
    tok = AutoTokenizer.from_pretrained(base_name)
    mdl = AutoModelForSequenceClassification.from_pretrained(ckpt).to(device).eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch):
            enc = tok(texts[i:i+batch], truncation=True, max_length=SEQ_LEN,
                      padding=True, return_tensors='pt').to(device)
            probs = torch.softmax(mdl(**enc).logits, dim=-1)[:, 1]
            out.extend(probs.cpu().tolist())
    return np.array(out)

texts = test['text'].tolist()
y = test['label'].to_numpy()
slices = ['obfuscated' if normalize(t) != t.strip().lower() else 'plain' for t in texts]
if 'lang' in test.columns:
    slices = [f'{s}/{l}' for s, l in zip(slices, test['lang'].fillna('?'))]

results = {}
for name in CANDIDATES:
    slug = name.split('/')[-1]
    s = score_checkpoint(f'artifacts/{slug}', name, texts)
    thr = evaluate.recommend_threshold(y, s)
    summary = evaluate.summarize(y, s, threshold=thr)
    results[slug] = (s, thr, summary)
    print(f"{slug}: F1={summary['f1']:.3f} PR-AUC={summary['pr_auc']:.3f} "
          f"R@FPR1%={summary['recall_at_fpr_1pct']:.3f} thr={thr}")
    for k, v in evaluate.sliced_metrics(y, s, slices, thr).items():
        print(f"   {k}: n={v['n']} " + (f"F1={v['f1']:.3f}" if 'f1' in v else f"acc={v['accuracy']:.3f}"))

WINNER = max(results, key=lambda k: results[k][2]['f1'])
print('\nWINNER:', WINNER, '(check size/latency too before shipping)')

In [ ]:
# full report + worst errors for the winner
s, thr, summary = results[WINNER]
errors = evaluate.error_table(texts, y, s)
report = evaluate.markdown_report(f'Transformer report — {WINNER}', summary,
                                  evaluate.threshold_sweep(y, s), errors,
                                  [f'- Recommended threshold: {thr}'])
report += '\n' + evaluate.sliced_section(evaluate.sliced_metrics(y, s, slices, thr))
Path(f'artifacts/{WINNER}_report.md').write_text(report)
print(report[:1500])

## 6. Convert winner to TFLite (int8) + parity gate

In [ ]:
!python -m samind_ml.export_transformer --checkpoint artifacts/$WINNER \
    --out artifacts/trigger_transformer.tflite --mode int8 --rep-data "$DATA" \
    --seq-len 128
!python -m samind_ml.parity --checkpoint artifacts/$WINNER \
    --tflite artifacts/trigger_transformer.tflite --data data/splits/val.csv

In [ ]:
# regenerate the shared golden vectors from the real vocab
!python -m samind_ml.make_goldens --vocab artifacts/vocab.txt \
    --out tests/data/golden_wordpiece.json
!cp artifacts/vocab.txt tests/data/vocab.txt
!python -m pytest tests/test_wordpiece_golden.py -q

## 7. Ship artifacts to Drive

In [ ]:
!cp artifacts/trigger_transformer.tflite artifacts/vocab.txt \
    artifacts/${WINNER}_report.md tests/data/golden_wordpiece.json "$OUT/"
!zip -qr "$OUT/${WINNER}_checkpoint.zip" artifacts/$WINNER
!ls -lh "$OUT"

## Shipping checklist

1. Parity cell printed **PASS** — if not, do not ship.
2. Test F1 ≥ 0.85 and R@FPR=1% ≥ 0.80 (task-card gates); obfuscated slice within 5 pp of plain.
3. Copy `trigger_transformer.tflite` **and** `vocab.txt` into `android/app/src/main/assets/` — they ship as a pair.
4. Replace `ml/tests/data/golden_wordpiece.json` (+ `vocab.txt`) in the repo with the regenerated ones; run the Kotlin golden test in CI.
5. Set `MODEL_THRESHOLD` in `TriggerClassifier.kt` to the recommended threshold from the report.
6. Compare false positives against the baseline report — the card wants 30%+ fewer.